# Assignment 6 - Object detection with YOLOv8n (transfer learning)
**Offline version** - runs without internet.
Needs `yolov8n.pt` and `datasets/coco128/` next to this notebook (see `00_download_once.ipynb`). `pip install ultralytics`.

In [ ]:
# ============================================================
# ASSIGNMENT 6: OBJECT DETECTION USING TRANSFER LEARNING
# Model: YOLOv8n | Dataset: COCO128
# Includes freezing, fine-tuning, evaluation and visualization
# ============================================================

# (pip install removed for offline use - install once with:  pip install ultralytics matplotlib)
import os
os.environ["YOLO_OFFLINE"] = "1"   # stop Ultralytics from trying to go online

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import ROOT, ASSETS
import yaml

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device)


# ============================================================
# 1. LOAD PRETRAINED MODEL
# ============================================================

# ---- OFFLINE FILES --------------------------------------------------------------
# 1) yolov8n.pt (6.5 MB): https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt
# 2) datasets/coco128/ (7 MB zip): https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip
#    -> unzip so you have  datasets/coco128/images/train2017  and  datasets/coco128/labels/train2017
# 3) Arial.ttf (for plots, optional): https://github.com/ultralytics/assets/releases/download/v0.0.0/Arial.ttf
# 00_download_once.ipynb fetches all of these for you.
WEIGHTS = Path("yolov8n.pt")
DATASET_DIR = Path("datasets/coco128").resolve()
RESULTS_DIR = Path("assignment6_results").resolve()

if not WEIGHTS.exists():
    raise FileNotFoundError("yolov8n.pt not found next to this notebook (see 00_download_once.ipynb).")
if not (DATASET_DIR / "images" / "train2017").exists():
    raise FileNotFoundError(f"COCO128 not found at {DATASET_DIR} (see 00_download_once.ipynb).")

# Build a local dataset yaml: same class names as coco128.yaml, but pointing at the local folder
# and with the 'download:' line removed so nothing is fetched.
_cfg = yaml.safe_load(open(ROOT / "cfg" / "datasets" / "coco128.yaml", encoding="utf-8"))
_cfg["path"] = str(DATASET_DIR)
_cfg.pop("download", None)
DATA_YAML = "coco128_local.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(_cfg, f, sort_keys=False)
# -------------------------------------------------------------------------------

model = YOLO(str(WEIGHTS))
print("Pretrained YOLOv8n model loaded.")


# ============================================================
# 2. STAGE 1: TRAIN WITH INITIAL LAYERS FROZEN
# ============================================================

print("\nStarting Stage 1: Frozen-layer training...")

model.train(
    data=DATA_YAML,
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=10,
    device=device,
    workers=0,
    amp=False,   # skips the AMP check, which would try to download a model
    project=str(RESULTS_DIR),
    name="stage1_frozen",
    exist_ok=True,
    plots=True
)

# Obtain the actual directory used by Ultralytics
stage1_dir = Path(model.trainer.save_dir)
stage1_path = stage1_dir / "weights" / "best.pt"

# Fall back to last.pt if best.pt is unavailable
if not stage1_path.exists():
    stage1_path = stage1_dir / "weights" / "last.pt"

if not stage1_path.exists():
    raise FileNotFoundError(
        f"No trained weights found in: {stage1_dir / 'weights'}"
    )

print("Stage 1 completed.")
print("Stage 1 weights:", stage1_path)


# ============================================================
# 3. STAGE 2: FINE-TUNE ALL LAYERS
# ============================================================

print("\nStarting Stage 2: Fine-tuning...")

model = YOLO(str(stage1_path))

model.train(
    data=DATA_YAML,
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=0,
    lr0=0.001,
    device=device,
    workers=0,
    amp=False,   # skips the AMP check, which would try to download a model
    project=str(RESULTS_DIR),
    name="stage2_finetune",
    exist_ok=True,
    plots=True
)

stage2_dir = Path(model.trainer.save_dir)
final_path = stage2_dir / "weights" / "best.pt"

if not final_path.exists():
    final_path = stage2_dir / "weights" / "last.pt"

if not final_path.exists():
    raise FileNotFoundError(
        f"No final weights found in: {stage2_dir / 'weights'}"
    )

print("Fine-tuning completed.")
print("Final model:", final_path)


# ============================================================
# 4. EVALUATE FINAL MODEL
# ============================================================

model = YOLO(str(final_path))

print("\nEvaluating final model...")

metrics = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=8,
    device=device,
    workers=0,
    project=str(RESULTS_DIR),
    name="evaluation",
    exist_ok=True,
    plots=True
)

print("\n========== EVALUATION RESULTS ==========")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")


# ============================================================
# 5. OBJECT DETECTION AND BOUNDING-BOX VISUALIZATION
# ============================================================

print("\nRunning object detection on a sample image...")

image_url = str(ASSETS / "bus.jpg")   # sample image bundled inside the ultralytics package

results = model.predict(
    source=image_url,
    conf=0.25,
    imgsz=640,
    device=device,
    save=False
)

# Draw bounding boxes, labels and confidence scores
annotated = results[0].plot()

# Convert BGR to RGB for Matplotlib
annotated = annotated[:, :, ::-1]

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.title("Object Detection Using Transfer Learning")
plt.axis("off")
plt.tight_layout()

output_image = "assignment6_object_detection.png"
plt.savefig(output_image, dpi=150, bbox_inches="tight")
plt.show()


# ============================================================
# 6. DISPLAY DETECTED OBJECTS
# ============================================================

print("\n========== DETECTED OBJECTS ==========")

boxes = results[0].boxes

if boxes is not None and len(boxes) > 0:
    for box in boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        coordinates = box.xyxy[0].tolist()

        print(
            f"Object: {model.names[class_id]} | "
            f"Confidence: {confidence:.2%} | "
            f"Bounding box: "
            f"{[round(v, 2) for v in coordinates]}"
        )
else:
    print("No objects detected.")


# ============================================================
# 7. FINAL OUTPUT
# ============================================================

print("\n========== ASSIGNMENT COMPLETED ==========")
print("Stage 1: Frozen-layer training")
print("Stage 2: Fine-tuning")
print("Final model:", final_path)
print("Detection image:", output_image)
print("Results directory:", stage2_dir)
print("Evaluation metrics and plots have been saved.")